# Лабораторная работа 7 Ансамбли моделей машинного обучения. Часть 2.

Панасюк Ксения ИУ5-64Б

Цель лабораторной работы: изучение ансамблей моделей машинного обучения.

Задание:

    Выберите набор данных (датасет) для решения задачи классификации или регресии.

    В случае необходимости проведите удаление или заполнение пропусков и кодирование категориальных признаков.

    С использованием метода train_test_split разделите выборку на обучающую и тестовую.

    Обучите следующие ансамблевые модели:
        одну из моделей группы стекинга.
        модель многослойного персептрона. По желанию, вместо библиотеки scikit-learn возможно использование библиотек TensorFlow, PyTorch или других аналогичных библиотек.
        (дополнительно) двумя методами на выбор из семейства МГУА (один из линейных методов COMBI / MULTI + один из нелинейных методов MIA / RIA) с использованием библиотеки gmdh.
        В настоящее время библиотека МГУА не позволяет решать задачу классификации !!!

    Оцените качество моделей с помощью одной из подходящих для задачи метрик. Сравните качество полученных моделей.


In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import StackingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, r2_score


In [2]:
california = fetch_california_housing(as_frame=True)
X = california.data
y = california.target

# Разделение выборки на обучающую и тестовую
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Масштабирование признаков 
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [3]:
# Словари для хранения результатов
results_r2 = {}
results_rmse = {}

def evaluate_model(model_name, y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    results_r2[model_name] = r2
    results_rmse[model_name] = rmse
    print(f"[{model_name}] R² Score: {r2:.4f} | RMSE: {rmse:.4f}")

In [4]:
# Обучение модели Стекинга
base_estimators = [
    ('ridge', Ridge(alpha=1.0)),
    ('rf', RandomForestRegressor(n_estimators=50, max_depth=10, random_state=42, n_jobs=-1))
]
stacking_model = StackingRegressor(
    estimators=base_estimators,
    final_estimator=LinearRegression()
)
stacking_model.fit(X_train_scaled, y_train)
y_pred_stack = stacking_model.predict(X_test_scaled)
evaluate_model("Stacking (Ridge + RF)", y_test, y_pred_stack)

[Stacking (Ridge + RF)] R² Score: 0.7735 | RMSE: 0.5448


In [5]:
# Обучение Многослойного персептрона 
mlp_model = MLPRegressor(
    hidden_layer_sizes=(64, 32),
    activation='relu',
    solver='adam',
    max_iter=300,
    random_state=42
)
mlp_model.fit(X_train_scaled, y_train)
y_pred_mlp = mlp_model.predict(X_test_scaled)
evaluate_model("MLP (Neural Network)", y_test, y_pred_mlp)


[MLP (Neural Network)] R² Score: 0.7907 | RMSE: 0.5237


In [6]:
print("\n--- Сводные результаты работы моделей ---")
df_results = pd.DataFrame({
    'R² Score (Выше -> Лучше)': results_r2,
    'RMSE (Ниже -> Лучше)': results_rmse
})
print(df_results)


--- Сводные результаты работы моделей ---
                       R² Score (Выше -> Лучше)  RMSE (Ниже -> Лучше)
Stacking (Ridge + RF)                  0.773483              0.544821
MLP (Neural Network)                   0.790684              0.523726
